# Atividade de vendas no Brasil

Notebook com os três entregáveis: NumPy, Pandas e Seaborn, usando o dataset `data/vendas_brasil_1.csv`.

In [ ]:
# Caixa 1
# Importando as bibliotecas que vou usar no notebook inteiro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../data/vendas_brasil_1.csv')
df.head()

## Entregável 1 NumPy

### A. Criando arrays

In [ ]:
# Caixa 2
# Pego as colunas numéricas do dataset e transformo cada uma em array do NumPy
# frete tem alguns valores faltantes, então preencho com 0 antes de virar array
# (assumo frete grátis quando não tem informação registrada)
preco_unitario = df['preco_unitario'].to_numpy()
custo_unitario = df['custo_unitario'].to_numpy()
quantidade = df['quantidade'].to_numpy()
desconto_pct = df['desconto_pct'].to_numpy()
receita_total = df['receita_total'].to_numpy()
frete = df['frete'].fillna(0).to_numpy()

print('preco_unitario:', preco_unitario[:5])
print('quantidade:', quantidade[:5])

### B. Cálculos com operações vetorizadas

In [ ]:
# Caixa 3
# Todos os cálculos aqui são vetorizados, sem for linha por linha
valor_bruto = preco_unitario * quantidade
valor_desconto = valor_bruto * (desconto_pct / 100)
custo_total = custo_unitario * quantidade
lucro_estimado = valor_bruto - valor_desconto - custo_total - frete

print('valor bruto médio:', valor_bruto.mean())
print('lucro estimado médio:', lucro_estimado.mean())

### C. Máscaras booleanas

In [ ]:
# Caixa 4
receita_media = receita_total.mean()

mask_acima_media = receita_total > receita_media
mask_prejuizo = lucro_estimado < 0
mask_desconto_alto = desconto_pct > 20

print('vendas acima da média:', mask_acima_media.sum())
print('vendas com prejuízo:', mask_prejuizo.sum())
print('vendas com desconto alto:', mask_desconto_alto.sum())

In [ ]:
# Caixa 5
# Dando uma olhada em algumas das vendas com prejuízo
df[mask_prejuizo].head()

### D. Agregações

In [ ]:
# Caixa 6
receita_media_geral = receita_total.mean()
maior_receita = receita_total.max()
menor_receita = receita_total.min()
lucro_medio = lucro_estimado.mean()
qtd_prejuizo = mask_prejuizo.sum()

print('receita média:', receita_media_geral)
print('maior receita:', maior_receita)
print('menor receita:', menor_receita)
print('lucro médio:', lucro_medio)
print('quantidade de vendas com prejuízo:', qtd_prejuizo)

**Conclusão do Entregável 1:** dá pra ver que a maioria das vendas fica perto da média de receita,
mas existe uma quantidade de vendas com prejuízo que provavelmente está ligada a descontos
muito altos. Isso já mostra que vale a pena olhar com mais calma a relação entre desconto
e lucro lá no entregável de Seaborn.

## Entregável 2 Pandas

### A. Primeiro olhar no dataset

In [ ]:
# Caixa 7
df.head()

In [ ]:
# Caixa 8
df.info()

In [ ]:
# Caixa 9
df.describe()

In [ ]:
# Caixa 10
df.shape

In [ ]:
# Caixa 11
df.isna().sum()

1. O dataset tem 37.617 linhas e 21 colunas.
2. Sim, existem dados faltantes em algumas colunas: canal_venda (50), vendedor (40),
margem_pct (82), frete (1.879), avaliacao_cliente (2.955) e prazo_entrega_dias (1.504).
3. As colunas numéricas são preco_unitario, custo_unitario, margem_pct, quantidade,
desconto_pct, receita_total, frete, avaliacao_cliente e prazo_entrega_dias.
4. As colunas categóricas são cidade, canal_venda, categoria, produto, vendedor e
fidelidade_cliente, além de colunas booleanas como devolucao, entrega_no_prazo e
alta_temporada.

### B. Limpeza e tratamento

In [ ]:
# Caixa 12
# A coluna de data vem misturando formatos (2022-04-13, 26-10-2023, 2022/09/20...),
# então uso format='mixed' com dayfirst=True pra dar conta de todos os formatos juntos
df['data_venda'] = pd.to_datetime(df['data_venda'], format='mixed', dayfirst=True, errors='coerce')

# A coluna canal_venda tem a mesma informação escrita de várias formas
# (App Mobile, APP Mobile, app mobile, SiteWeb, Loja Fisica, loja física...)
def padronizar_canal(valor):
    if pd.isna(valor):
        return 'Desconhecido'
    chave = valor.strip().lower().replace(' ', '')
    mapa = {
        'appmobile': 'App Mobile',
        'siteweb': 'Site Web',
        'lojafisica': 'Loja Física',
        'lojafísica': 'Loja Física',
        'marketplace': 'Marketplace',
        'televendas': 'Televendas',
    }
    return mapa.get(chave, valor.strip().title())

df['canal_venda'] = df['canal_venda'].apply(padronizar_canal)

# A coluna cidade também tem várias formas diferentes pro mesmo lugar
# (espaço sobrando, abreviação tipo SP/RJ/BH, erro de digitação tipo Curtiba/Slavador)
def padronizar_cidade(valor):
    if pd.isna(valor):
        return 'Desconhecido'
    chave = valor.strip().lower()
    substituicoes = str.maketrans('áàãâéêíóõôú', 'aaaaeeiooou')
    chave = chave.translate(substituicoes)
    mapa = {
        'rio de janeiro': 'Rio de Janeiro', 'rio': 'Rio de Janeiro', 'rj': 'Rio de Janeiro',
        'sao paulo': 'São Paulo', 's. paulo': 'São Paulo', 'sp': 'São Paulo',
        'belo horizonte': 'Belo Horizonte', 'b. horizonte': 'Belo Horizonte', 'bh': 'Belo Horizonte',
        'salvador': 'Salvador', 'slavador': 'Salvador',
        'curitiba': 'Curitiba', 'curtiba': 'Curitiba',
        'recife': 'Recife', 'fortaleza': 'Fortaleza', 'manaus': 'Manaus',
        'goiania': 'Goiânia', 'porto alegre': 'Porto Alegre',
    }
    return mapa.get(chave, valor.strip())

df['cidade'] = df['cidade'].apply(padronizar_cidade)

# Tratando dados faltantes: frete faltante vira 0, vendedor faltante vira 'Desconhecido'
df['frete'] = df['frete'].fillna(0)
df['vendedor'] = df['vendedor'].fillna('Desconhecido')

df.isna().sum()

Preenchi o frete faltante com 0 porque assumo que, quando não tem valor registrado, a
entrega saiu sem custo de frete pro cliente. Já o vendedor faltante virou "Desconhecido"
porque não tem como recuperar quem fez a venda só olhando as outras colunas. Padronizei
canal_venda e cidade porque as duas apareciam escritas de vários jeitos diferentes
(espaço sobrando, letra maiúscula/minúscula, abreviação, erro de digitação), o que
atrapalharia qualquer agrupamento por canal ou por cidade.

### C. Criação de novas colunas

In [ ]:
# Caixa 13
df['ano_venda'] = df['data_venda'].dt.year
df['mes_venda'] = df['data_venda'].dt.month
df['lucro_total'] = (df['preco_unitario'] * df['quantidade'] * (1 - df['desconto_pct'] / 100)) - (df['custo_unitario'] * df['quantidade']) - df['frete']
df['receita_liquida'] = df['receita_total'] - df['frete']

# Essa coluna aqui uso apply/lambda pra classificar a receita em faixas
# (olhei o describe() da receita_total antes de escolher esses cortes)
def classificar_faixa(valor):
    if valor < 500:
        return 'baixa'
    elif valor < 2000:
        return 'media'
    else:
        return 'alta'

df['faixa_receita'] = df['receita_total'].apply(lambda x: classificar_faixa(x))

df[['ano_venda', 'mes_venda', 'lucro_total', 'receita_liquida', 'faixa_receita']].head()

### D. Filtros

In [ ]:
# Caixa 14
vendas_receita_alta = df[df['receita_total'] > 2000]
print('vendas com receita acima de 2000:', len(vendas_receita_alta))

vendas_devolvidas = df[df['devolucao'] == True]
print('vendas com devolução:', len(vendas_devolvidas))

vendas_eletronicos = df[df['categoria'] == 'Eletrônicos']
print('vendas de eletrônicos:', len(vendas_eletronicos))

Boa parte das vendas com receita alta se concentra em poucas categorias, e a quantidade
de devoluções é bem menor que o total de vendas, o que já era esperado.

### E. Agrupamentos com groupby

In [ ]:
# Caixa 15
receita_por_categoria = df.groupby('categoria')['receita_total'].sum().sort_values(ascending=False)
receita_por_categoria

In [ ]:
# Caixa 16
vendas_por_canal = df.groupby('canal_venda')['quantidade'].sum().sort_values(ascending=False)
vendas_por_canal

In [ ]:
# Caixa 17
receita_por_cidade = df.groupby('cidade')['receita_total'].sum().sort_values(ascending=False)
receita_por_cidade.head()

In [ ]:
# Caixa 18
lucro_por_categoria = df.groupby('categoria')['lucro_total'].sum().sort_values(ascending=False)
lucro_por_categoria

A categoria com maior receita não é necessariamente a de maior lucro, o que sugere que
algumas categorias vendem muito mas com margem mais apertada.

### F. Merge

In [ ]:
# Caixa 19
# Tabela auxiliar relacionando algumas cidades a regiões
tabela_regioes = pd.DataFrame({
    'cidade': ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Salvador', 'Fortaleza',
               'Recife', 'Porto Alegre', 'Curitiba', 'Manaus', 'Goiânia'],
    'regiao': ['Sudeste', 'Sudeste', 'Sudeste', 'Nordeste', 'Nordeste',
               'Nordeste', 'Sul', 'Sul', 'Norte', 'Centro-Oeste']
})

df_com_regiao = df.merge(tabela_regioes, on='cidade', how='left')
df_com_regiao['regiao'] = df_com_regiao['regiao'].fillna('Outra')

receita_por_regiao = df_com_regiao.groupby('regiao')['receita_total'].sum().sort_values(ascending=False)
receita_por_regiao

**Conclusão do Entregável 2:** depois de limpar e organizar os dados, dá pra ver que a
receita se concentra em poucas categorias e cidades, e que a região Sudeste puxa a maior
parte da receita total, o que faz sentido dado o tamanho do mercado nessas cidades.

## Entregável 3 Seaborn

### A. Gráfico de barras

In [ ]:
# Caixa 20
receita_categoria = df.groupby('categoria')['receita_total'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=receita_categoria.index, y=receita_categoria.values, linewidth=0.8, edgecolor='black')
plt.title('Receita total por categoria')
plt.xlabel('Categoria')
plt.ylabel('Receita total')
plt.xticks(rotation=45)
sns.despine()
plt.tight_layout()
plt.show()

O gráfico mostra que a receita não é distribuída de forma igual entre as categorias,
algumas poucas concentram a maior parte do faturamento.

### B. Gráfico de linha

In [ ]:
# Caixa 21
receita_mensal = df.groupby(df['data_venda'].dt.to_period('M'))['receita_total'].sum()
receita_mensal.index = receita_mensal.index.astype(str)

plt.figure(figsize=(9, 5))
sns.lineplot(x=receita_mensal.index, y=receita_mensal.values, marker='o', linewidth=1.5)
plt.title('Receita ao longo dos meses')
plt.xlabel('Mês')
plt.ylabel('Receita total')
plt.xticks(rotation=45)
sns.despine()
plt.tight_layout()
plt.show()

Dá pra observar se existe algum mês com pico de vendas, o que pode indicar sazonalidade,
tipo datas comemorativas.

### C. Gráfico de dispersão

In [ ]:
# Caixa 22
amostra = df.sample(n=min(300, len(df)), random_state=42)

plt.figure(figsize=(7, 5))
sns.scatterplot(data=amostra, x='desconto_pct', y='receita_total', alpha=0.6)
plt.title('Relação entre desconto e receita (amostra)')
plt.xlabel('Desconto (%)')
plt.ylabel('Receita total')
sns.despine()
plt.tight_layout()
plt.show()

Não parece existir uma relação muito forte entre desconto e receita nessa amostra, o que
sugere que o desconto sozinho não explica bem o tamanho da receita da venda.

### D. Mapa de calor

In [ ]:
# Caixa 23
colunas_numericas = ['preco_unitario', 'custo_unitario', 'quantidade', 'desconto_pct', 'receita_total', 'frete', 'lucro_total']
correlacao = df[colunas_numericas].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(correlacao, annot=True, fmt='.2f', linewidths=0.5)
plt.title('Correlação entre variáveis numéricas')
plt.tight_layout()
plt.show()

**Conclusão final:** os gráficos mostram que a receita está concentrada em poucas
categorias e cidades, que existe alguma variação de receita ao longo dos meses, e que o
desconto isoladamente não parece ser o principal fator que explica a receita das vendas.
O lucro, por outro lado, tem relação mais direta com preço e custo unitário, como era de
se esperar.